# 1단계 — 파이썬 & pandas for 퀀트

**목표:** 가격 데이터를 자유롭게 자르고, 변환하고, 요약할 수 있게 되기.
백테스트에 필요한 pandas 기능만 골라서 다룹니다.

| 절 | 내용 | 퀀트에서 쓰이는 곳 |
|---|---|---|
| 1 | DataFrame 구조, 인덱싱 | 모든 것의 기본 |
| 2 | 날짜 인덱스로 자르기 | 기간별 분석 |
| 3 | 수익률 계산 | 성과 측정 |
| 4 | 리샘플링 (일→주/월) | 월간 리밸런싱 |
| 5 | 롤링 윈도우 | 이동평균, 변동성 |
| 6 | `shift`와 미래참조 편향 | 백테스트의 핵심 함정 |
| 7 | 여러 종목 합치기 | 포트폴리오 |
| 8 | groupby로 요약 | 연도별 성과 |

각 절 끝에 ✏️ **연습문제**가 있습니다. 정답은 `01_pandas_solutions.ipynb`에 있어요 — 먼저 직접 풀어보세요!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from quant_study.data import load_price

plt.rcParams['figure.figsize'] = (10, 4)
import platform
plt.rcParams['font.family'] = {'Darwin': 'AppleGothic', 'Windows': 'Malgun Gothic'}.get(platform.system(), 'NanumGothic')  # 한글 폰트
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.float_format', '{:,.4f}'.format)

## 1. DataFrame 구조와 인덱싱

`load_price`는 처음 한 번만 인터넷에서 받고, 이후에는 `data/` 폴더의 CSV를 읽습니다.

In [ ]:
df = load_price('005930', start='2020-01-01')   # 삼성전자
df.head()

In [ ]:
df.info()   # 행 개수, 컬럼 타입, 결측치 여부를 한눈에

- **인덱스(index)**: 행 이름. 여기서는 날짜(`DatetimeIndex`)
- **컬럼(columns)**: `Open, High, Low, Close, Volume, Change`
- 컬럼 하나를 꺼내면 **Series**, 여러 개면 **DataFrame**

In [ ]:
close = df['Close']            # Series
print(type(close))
df[['Close', 'Volume']].tail(3)   # DataFrame

`loc`는 **이름(라벨)** 으로, `iloc`는 **위치(숫자)** 로 고릅니다.

In [ ]:
print(df.iloc[0])                 # 첫 번째 행
print(df.iloc[-1]['Close'])       # 마지막 종가
df.loc['2024-01-02', 'Close']     # 특정 날짜의 종가

**조건으로 거르기(boolean indexing)** — 조건식은 True/False Series를 만들고, 그걸로 행을 고릅니다.

In [ ]:
big_volume = df['Volume'] > df['Volume'].quantile(0.99)   # 거래량 상위 1%
df[big_volume].head()

In [ ]:
# 조건 여러 개는 &(그리고), |(또는), 각 조건은 괄호로 감싸기
df[(df['Change'] > 0.05) | (df['Change'] < -0.05)].shape   # 하루 ±5% 넘게 움직인 날 수

### ✏️ 연습문제 1
1. 전체 기간 중 종가(`Close`)가 가장 높았던 날짜와 그 가격을 구하세요. (힌트: `idxmax()`)
2. 시가보다 종가가 높게 끝난 날(양봉)은 전체의 몇 %인가요?

In [ ]:
# 1-1


In [ ]:
# 1-2


## 2. 날짜 인덱스로 자르기

`DatetimeIndex`는 문자열로 기간을 자를 수 있습니다. 끝 날짜도 **포함**된다는 점에 주의하세요.

In [ ]:
df.loc['2023']                      # 2023년 전체
df.loc['2023-03':'2023-05'].shape   # 3~5월

In [ ]:
print(df.index.year[:3], df.index.month[:3], df.index.dayofweek[:3])  # 0=월요일

### ✏️ 연습문제 2
1. 2022년 한 해 동안의 최고 종가와 최저 종가를 구하세요.
2. 요일별 평균 거래량을 구하세요. (힌트: `df.index.dayofweek`로 거른 뒤 평균 — 8절의 groupby로 더 쉽게 할 수도 있어요)

In [ ]:
# 2-1


In [ ]:
# 2-2


## 3. 수익률 계산

퀀트에서는 가격보다 **수익률**을 주로 다룹니다. (가격 수준이 다른 종목끼리 비교할 수 있으니까요)

- 단순 수익률: $r_t = \frac{P_t}{P_{t-1}} - 1$ → `pct_change()`
- 로그 수익률: $\ln(P_t / P_{t-1})$ → 기간별로 **더할** 수 있어 통계 계산에 편리
- 누적 수익률: $\prod (1 + r_t) - 1$ → `(1 + r).cumprod() - 1`

In [ ]:
ret = df['Close'].pct_change()
log_ret = np.log(df['Close'] / df['Close'].shift(1))
pd.DataFrame({'ret': ret, 'log_ret': log_ret}).head()

첫 행이 `NaN`인 이유: 전날 가격이 없으니까요. `dropna()`로 지우거나 그대로 두면 대부분의 함수가 알아서 무시합니다.

In [ ]:
cum_ret = (1 + ret.fillna(0)).cumprod() - 1
cum_ret.plot(title='삼성전자 누적 수익률')
plt.axhline(0, color='gray', lw=0.8);

In [ ]:
# 검산: 누적 수익률의 마지막 값 == 마지막 가격 / 첫 가격 - 1
print(cum_ret.iloc[-1], df['Close'].iloc[-1] / df['Close'].iloc[0] - 1)

### ✏️ 연습문제 3
1. 일간 수익률이 가장 컸던 날과 가장 작았던 날은?
2. 로그 수익률을 모두 더한 뒤 `np.exp(합) - 1`을 계산해, 누적 수익률과 같은지 확인하세요. 왜 같을까요?

In [ ]:
# 3-1


In [ ]:
# 3-2


## 4. 리샘플링 — 일봉을 주봉·월봉으로

`resample(규칙)` 뒤에 어떻게 합칠지(집계 함수)를 붙입니다.

| 규칙 | 의미 |
|---|---|
| `'W'` | 주 (일요일 기준) |
| `'ME'` | 월말 |
| `'QE'` | 분기말 |
| `'YE'` | 연말 |

In [ ]:
monthly_close = df['Close'].resample('ME').last()   # 매월 마지막 종가
monthly_ret = monthly_close.pct_change()
monthly_ret.tail()

In [ ]:
# 월봉 OHLCV 만들기: 컬럼마다 집계 방법이 다름
monthly = df.resample('ME').agg({
    'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'
})
monthly.tail(3)

### ✏️ 연습문제 4
1. 주간 수익률(`'W'`)을 구하고, 주간 수익률이 플러스였던 주의 비율을 구하세요.
2. 연도별 수익률(`'YE'`)을 막대그래프로 그리세요. (`.plot.bar()`)

In [ ]:
# 4-1


In [ ]:
# 4-2


## 5. 롤링 윈도우 — 이동평균과 변동성

`rolling(n)`은 최근 n개 행을 묶어서 계산합니다. 처음 n-1개는 데이터가 모자라 `NaN`이 됩니다.

In [ ]:
df['MA20'] = df['Close'].rolling(20).mean()
df['MA60'] = df['Close'].rolling(60).mean()
df.loc['2024', ['Close', 'MA20', 'MA60']].plot(title='삼성전자 이동평균 (2024)');

**변동성(volatility)** = 수익률의 표준편차. 일간 값을 1년 기준으로 바꾸려면 $\sqrt{252}$를 곱합니다 (1년 ≈ 252 거래일).

In [ ]:
vol_20 = ret.rolling(20).std() * np.sqrt(252)
vol_20.plot(title='20일 롤링 연환산 변동성');

In [ ]:
# 누적 최고점 대비 하락폭 = 낙폭(drawdown)
peak = df['Close'].cummax()
drawdown = df['Close'] / peak - 1
drawdown.plot(title='낙폭 (Drawdown)', color='tab:red')
print('최대 낙폭(MDD):', drawdown.min())

### ✏️ 연습문제 5
1. 5일, 120일 이동평균을 구하고 2023년 구간을 종가와 함께 그리세요.
2. 연환산 변동성이 가장 높았던 날짜는? 그 무렵 무슨 일이 있었는지 찾아보세요.

In [ ]:
# 5-1


In [ ]:
# 5-2


## 6. `shift`와 미래참조 편향 ⚠️

**백테스트에서 가장 흔한 실수**: 오늘 종가로 만든 신호로 오늘 수익률을 먹는 것.
오늘 종가를 보고 결정하면, 실제로 매매할 수 있는 건 **내일부터**입니다.

`shift(1)`은 값을 한 칸 아래(미래 방향)로 밀어서 "어제의 값"을 오늘 행에 둡니다.

In [ ]:
tmp = pd.DataFrame({'Close': df['Close'], 'prev_close': df['Close'].shift(1)})
tmp.head()

간단한 전략으로 차이를 확인해 봅시다: **종가가 20일 이동평균 위면 보유, 아니면 현금.**

In [ ]:
signal = (df['Close'] > df['MA20']).astype(int)   # 1=보유, 0=현금

wrong = (1 + ret * signal).cumprod()             # ❌ 오늘 신호로 오늘 수익
right = (1 + ret * signal.shift(1)).cumprod()    # ✅ 어제 신호로 오늘 수익
buyhold = (1 + ret).cumprod()

pd.DataFrame({'미래참조(틀림)': wrong, '올바름': right, '단순보유': buyhold}).plot(logy=True, title='미래참조 편향의 위력');

'틀린' 버전이 비현실적으로 좋습니다. **백테스트 결과가 너무 좋으면 먼저 shift를 의심하세요.**

### ✏️ 연습문제 6
1. 위 전략을 MA20 대신 **MA60**으로 바꿔 올바른 버전의 최종 누적 수익률을 구하세요.
2. 올바른 버전에서 보유 중이었던 날의 비율은? (`signal.shift(1)`의 평균)

In [ ]:
# 6-1


In [ ]:
# 6-2


## 7. 여러 종목 합치기

`pd.concat(..., axis=1)`로 종가를 옆으로 붙이면 날짜 기준으로 자동 정렬됩니다.

In [ ]:
tickers = {'삼성전자': '005930', 'SK하이닉스': '000660', 'KOSPI': 'KS11'}
prices = pd.concat({name: load_price(code, start='2020-01-01')['Close'] for name, code in tickers.items()}, axis=1)
prices.tail()

In [ ]:
print(prices.isna().sum())      # 휴장일 차이 등으로 빈 칸이 생길 수 있음
prices = prices.ffill().dropna()  # 앞의 값으로 채우고, 그래도 남은 앞부분은 삭제

In [ ]:
# 시작점을 1로 맞춰(정규화) 비교
(prices / prices.iloc[0]).plot(title='정규화 가격 비교');

In [ ]:
rets = prices.pct_change().dropna()
rets.corr()   # 수익률 상관계수

### ✏️ 연습문제 7
1. 세 자산의 연환산 변동성을 한 번에 구하세요. (DataFrame에 `.std()`를 쓰면 컬럼별로 계산됩니다)
2. 삼성전자 50% + SK하이닉스 50%를 **매일 리밸런싱**하는 포트폴리오의 누적 수익률을 구해 KOSPI와 비교해 그리세요.

In [ ]:
# 7-1


In [ ]:
# 7-2


## 8. groupby로 요약하기

"~별로" 계산하고 싶을 때 `groupby`를 씁니다.

In [ ]:
yearly = rets.groupby(rets.index.year).apply(lambda r: (1 + r).prod() - 1)
yearly   # 연도별 수익률

In [ ]:
# 월별 평균 수익률 — 계절성이 있을까?
rets['삼성전자'].groupby(rets.index.month).mean().plot.bar(title='월별 평균 일간 수익률 (삼성전자)');

### ✏️ 연습문제 8
1. 요일별(`dayofweek`) 삼성전자 평균 수익률과 상승 확률을 한 표로 만드세요. (힌트: `.agg(['mean', lambda r: (r > 0).mean()])`)
2. 연도별 **최대 낙폭(MDD)** 을 세 자산 모두에 대해 구하세요.

In [ ]:
# 8-1


In [ ]:
# 8-2


## 🎯 1단계 종합 과제

지금까지 배운 것만으로 아래 **성과 요약표**를 만드는 함수 `summary(prices)`를 작성하세요.
입력은 종가 DataFrame(컬럼=종목), 출력은 종목별로 아래 지표가 담긴 DataFrame입니다.

| 지표 | 계산 |
|---|---|
| 총수익률 | 마지막/처음 - 1 |
| CAGR (연평균 성장률) | $(1+\text{총수익률})^{252/N} - 1$, N=거래일 수 |
| 연환산 변동성 | 일간 수익률 std × √252 |
| MDD | 낙폭의 최솟값 |

→ 이 함수는 3단계 백테스트에서 그대로 재사용합니다!

In [ ]:
def summary(prices: pd.DataFrame) -> pd.DataFrame:
    ...

# summary(prices)